# Lektion 13 - Agenthukommelse med Cognee Knowledge Graphs


## Setup

Denne notesbog demonstrerer, hvordan man bygger en intelligent **kodeassistent** med vedvarende hukommelse ved hjælp af [**Cognee**](https://www.cognee.ai/) vidensgrafer og **Microsoft Agent Framework** (MAF).

Cognee omdanner ustruktureret tekst til en struktureret, forespørgbar vidensgraf understøttet af vektorindlejringer — hvilket giver din agent en rig, relationsbevidst langtidshukommelse.

### Det vil du lære
1. **Byg vidensgrafer**: Omdan udviklerprofiler og bedste praksis til struktureret, forespørgbar viden.
2. **Integrer Cognee med MAF**: Brug `@tool` funktioner til at lade en MAF-agent forespørge Cognee's vidensgraf.
3. **Sessionsbevidste samtaler**: Bevar kontekst på tværs af flere spørgsmål i samme session.
4. **Langtidshukommelse**: Gem vigtig viden på tværs af sessioner og hent den i nye samtaler.

### Forudsætninger
- Python 3.9+
- Redis kørende lokalt (`docker run -d -p 6379:6379 redis`) til sessionstyring
- En LLM API-nøgle (f.eks. OpenAI) — sæt `LLM_API_KEY` i `.env`
- `CACHING=true` i `.env` (kræves til Cognee-sessioner)
- Et Azure AI Foundry-projekt med en udrullet chatmodel
- `AZURE_AI_PROJECT_ENDPOINT` og `AZURE_AI_MODEL_DEPLOYMENT_NAME` i `.env`
- Azure CLI autentificeret (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Typer af agenthukommelse

Denne notesbog udforsker de samme tre hukommelsestyper som i hovednotesbogen til lektion 13, men bruger Cognee som backend for langtids hukommelsen:

| Hukommelsestype | Mekanisme | Levetid |
|---|---|---|
| **Arbejds-** | `agent.create_session()` (MAF) | Enkelt samtale |
| **Korttids-** | Cognee session cache (Redis) | Enkel session |
| **Langtids-** | Cognee knowledge graph + vektorer | Permanent |

### Cognees hukommelsesarkitektur  
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Forbered Cognee Lageret


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Del 1 — Opbygning af Vidensbasen

Vi indsamler tre typer data for at skabe en omfattende vidensbase til vores kodeassistent:

1. **Udviklerprofil** — personlig ekspertise og teknisk baggrund  
2. **Python Bedste Praksis** — Zen of Python med praktiske retningslinjer  
3. **Historiske Samtaler** — tidligere Q&A-sessioner mellem udviklere og AI-assistenter


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Visualiser Vidensgrafen

Cognee kan gengive en interaktiv HTML-visualisering af enhederne og relationerne, den har udtrukket.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Berig Hukommelsen med Memify

`memify()` analyserer vidensgrafen og genererer intelligente regler — identificerer mønstre, bedste praksis og relationer mellem begreber.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Del 2 — MAF-agent med Cognee-værktøjer

Nu opretter vi en MAF-agent, der kan forespørge Cognees vidensgraf via `@tool`-funktioner. Dette giver agenten mulighed for at udnytte den fulde kraft af graf-bevidst semantisk søgning samtidig med, at den opretholder samtalekontekst gennem sessioner.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Arbejdshukommelse med sessioner

`AgentSession` (oprettet via `agent.create_session()`) giver arbejdshukommelse inden for en session. agenten kan referere tilbage til tidligere beskeder samtidig med, at den også kan forespørge Cognees langsigtede vidensgraf.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Ny session — Langtidshukommelse bevares

Når du starter en ny session, ryddes arbejdshukommelsen, men Cognee-vidensgrafen er stadig tilgængelig. Agenten kan hente den samme langsigtede viden i en helt ny samtale.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Oversigt

I denne notesbog byggede du en kodningsassistent, der kombinerer **MAF's arbejdshukommelse** (`agent.create_session()`) med **Cognee's langsigtede vidensgraf**.

### Hvad du har lært
1. **Opbygning af vidensgraf**: Cognee indtager ustruktureret tekst og bygger en graf + vektormemory.
2. **Berigelse af graf med memify**: Udledte fakta og rigere relationer oven på din eksisterende graf.
3. **MAF + Cognee integration**: `@tool` funktioner lader en MAF-agent forespørge Cognee's graf naturligt.
4. **Arbejdshukommelse + langsigtet hukommelse**: `AgentSession` (via `agent.create_session()`) giver sessionskontekst, mens Cognee leverer vedvarende viden.
5. **Filtreret søgning med NodeSets**: Målret specifikke undergrupper af vidensgrafen (f.eks. kun principper).

### Centrale pointer
- **Cognee** omdanner rå tekst til struktureret, relationsbevidst hukommelse — mere kraftfuld end en flad vektorlager.
- **`@tool` funktioner** forbinder MAF-agenter og eksterne videnssystemer på en ren måde.
- **`AgentSession`** (via `agent.create_session()`) holder kontekst per samtale adskilt fra langtidsviden.
- Den samme vidensgraf tjener flere sessioner og agenter.

### Virkelige anvendelser
- **Udvikler-copiloter**: Kodegennemgang, hændelsesanalyse, arkitektassistenter
- **Kundeorienterede copiloter**: Supportagenter over produktdokumentation, FAQ’er og CRM-noter
- **Interne ekspertcopiloter**: Politik-, juridiske- eller sikkerhedsassistenter der ræsonnerer over retningslinjer
- **Forenede datalag**: Kombiner strukturerede og ustrukturerede data i én forespørgbar graf

### Næste skridt
- Eksperimenter med tidsmæssig bevidsthed i Cognee
- Definer en OWL-ontologi for domænespecifik grafkvalitet
- Tilføj brugerfeedbacksløjfer for at forbedre opslag over tid
- Skaler til multi-agent systemer, der deler samme Cognee hukommelseslag


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi stræber efter nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det oprindelige dokument på dets modersmål bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
